# Pipelines

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
from sklearn import set_config
set_config(transform_output = "pandas")

> **"If you aren’t using pipelines you’re probably doing [Scikit-Learn] wrong."** - [Andreas Muller, Core Developer of Scikit-learn ](https://towardsdatascience.com/want-to-truly-master-scikit-learn-2-essential-tips-from-the-official-developer-himself-dada6ff56b99)

<img src="images/Data_Processing/sklearn-pipe.png" style="display: block;margin-left: auto;margin-right: auto;width: 600px"/>

#### What is a pipeline?

Whilst the statement above was probably an exaggeration, they are a great way to keep your code clean, consistent and mistake-free. 

Pipelines encapsulate all the preprocessing steps (feature selections, scaling, encoding of variables and so on), as well as the final model, into a single Scikit-Learn estimator. 

- Pipelines simplify and automate many steps in preprocessing and model training. 
- They give your workflow order and make it easier to read and understand. Later we will see how they can also be very useful during model optimization. 
- In addition to this including preprocessing as part of our model pipeline we can **avoid data leaks**.

Let's first **re-create** our train and test set, and the Column Transformer that was made in the previous notebook.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

titanic_df = pd.read_csv('data/titanic.csv')

def drop_unwanted_cols(df, cols):
    return df.drop(columns=cols)

def create_Xy(df, target='survived'):
    df = df.reset_index(drop=True)
    return df.drop(columns=[target]), df[target]

unwanted_cols = ['embarked', 'sex', 'adult_male',
                 'deck', 'alive', 'class']

X, y = (
    titanic_df
    .pipe(drop_unwanted_cols, cols=unwanted_cols)
    .dropna()
    .pipe(create_Xy)
)

# Create train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=111)

#### Preprocessing without a Pipeline

Let's see how you would create a model with one-hot encoding and a scaling step in between, _without_ using a pipeline. 

In [ ]:
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# List categorical columns
categorical_columns = ['who', 'embark_town']

# One-hot encode
column_transformer = ColumnTransformer([
    ('one_hot_encoder', OneHotEncoder(sparse_output=False, drop='first'), categorical_columns)
    ], remainder="passthrough")

X_train_encoded = column_transformer.fit_transform(X_train)
X_test_encoded = column_transformer.transform(X_test)

# Scaling 
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

# Model
model = SVC()

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
accuracy_score(y_test, y_pred)

In the current approach, this is possible, but it requires a lot of manual typing and effort. Notice in the example below that we must remember to create the new `_scaled` variables with our `_encoded` data **and** use those new variables in our fitting step? It's easy for little mistakes to slip in! Additionally, it's not very easy to quickly test if scaling improves our scores. 

#### Using a Pipeline instead

Let's rewrite this code as a pipeline.

A Scikit-Learn `Pipeline` simply requires us to specify a number of steps and what should happen at each of them. In our case we are going to add our one-hot encoding and support vector machine model. For simplicity, we are here refering to the `column_transformer` we created earlier. 

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

pipeline = Pipeline(steps=[
    ('onehot', column_transformer), 
    ('model', SVC())
])

Our pipeline has been created and now contains two steps: one-hot encoding the data and the support vector machine model. 

You could however very easily add a scaler to the Pipeline:

In [ ]:
pipeline = Pipeline(steps=[
    ('onehot', column_transformer), 
    ('scaler', MinMaxScaler()),
    ('model', SVC())
])

We can use our pipeline in the same way that we previously used our model: by fitting our data to it and using it to create predictions. 

In [ ]:
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

What happened here? When we called `pipeline.fit`, we simply had to pass it our original (**not** one-hot encoded) training and test data. The first step in our pipeline contains the column transformer that does our one-hot encoding, so our original data (`X_train`) is used to fit the column transformer **and** transform our train data into what we previously called `X_train_encoded`. Then that output is used for the next step: the model. 

Now, what happens during testing? When we called `pipeline.predict` on our test data, the data undergoes those same steps: first using the column transformer fitted on the train data to create what we previously called `X_test_encoded` and then using that output for our modelling step.

Note that it is possible to access the named stages of our pipelines.

In [ ]:
pipeline['scaler']

In [ ]:
pipeline['onehot'].get_feature_names_out()
# pipeline['onehot'].get_feature_names() # for older sklearn versions

With pipelines, adding a (preprocessing) step only involves adding a single line. By using a pipeline, the data is handled appropriately and no mistakes can slip in. This prevents data leakage and ensures your code is clean and secure

The beauty of sklearn pipelines is in the fact that it will learn how to scale the data based on the `X_train` features and then automatically use it when we get predictions from `X_test`. So in practice it also removes some preprocessing efforts and unnecessary risks away from us.

## <mark> Exercise: Pipeline </mark>
Try to add `PolynomialFeatures` to the pipeline. Experiment with:
- No Polynomial Features
- Polynomial Features with no additional settings
- Polynomial Features with `interaction_only` set to True

Click [here](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) for documentation on polynomial features. 

_Hint_: Make sure you use Polynomial Features _after_ one-hot encoding and _before_ scaling. 

In [ ]:
# add your code here


**<font color='green'> Bonus Challenge: During what step </font>** 

Why would we want to create polynomial features _after_ one-hot encoding and _before_ scaling? 

In [ ]:
# %load answers/06_Data_Processing/polynomial_features.py

## Conclusions

Scikit-Learn can be a very powerful tool to deal with machine learning problems including data splitting, preprocessing, model training and model selection. Its simple interface and detailed documentation allow it to be used even by users with little experience.

Its advantages include:
* A huge variety of implemented models with sensible defaults, such as KMeans, SVC or RandomForest
* Supports both pandas and numpy as data inputs
* Data preprocessing techniques such as Scaler, OneHotEncoding, etc. 
* Helpful tools such as train_test_split and pipelines
* Consistent implementation and API which makes it easy to extend with your own implemented building blocks (scorers, preprocessing techniques, models, etc.)

Caveat: Scikit-Learn lacks tools to work with deep learning. For this purpose you would rather use deep learning libraries such as PyTorch or Tensorflow/Keras.